# WO22 — L08 HYDE build

Builds the two L08 HYDE tables needed for HYDE choropleth at Level 08:

- **`temporal.hyde_basin08_weights`** — crosswalk of basin08 × hyde_cells, same schema as
  `hyde_basin06_weights` (WO17). One-time ST_Intersection build.
- **`temporal.hyde_basin08_steps`** — pre-aggregated `(hybas_id, step_idx, *_frac)` table,
  same schema as `hyde_basin06_steps` (WO18). Per-step loop over 128 HYDE time steps.

**L08 vs L06 expected differences:**
- 190,675 basins (11.6× L06's 16,397); but L08 basins are small (~19 HYDE cells/basin avg
  vs L06's ~173), so crosswalk rows are only ~3.6M vs L06's 2.82M.
- Steps table: ~24M rows (190k × 128) vs L06's 2.08M.
- Per-request query time: estimated ~0.38s vs L06's 0.033s — well within the 500ms threshold.

Both cells 4 and 6 are idempotent (skip if the table already exists).

**Gate:** per-request time <500ms; unit guard passes; row counts match expected.
After this notebook, the route change in `routes.py` and UI wiring complete Stage 2b.

In [1]:
# Cell 1
import warnings
warnings.filterwarnings('ignore', message='pandas only supports SQLAlchemy')

import time
import numpy as np
import pandas as pd
from pathlib import Path
from scripts.shared import db_utils

conn = db_utils.db_connect()

ROOT = Path(db_utils.__file__).parent.parent.parent
OUT  = ROOT / 'output' / 'edop' / 'surface'
OUT.mkdir(parents=True, exist_ok=True)

print('Connected. Output:', OUT)

Connected. Output: /Users/karlg/Documents/repos/_edops/output/edop/surface


## 1 — Prerequisites

In [2]:
# Cell 2 — Verify prerequisites
# • GIST index on hyde_cells.geom (crosswalk build needs ST_Intersects)
# • basin08 has geom + sub_area columns
# • L06 reference tables exist (confirms HYDE framework is in place)

idx_df = pd.read_sql("""
    SELECT indexname FROM pg_indexes
    WHERE tablename = 'hyde_cells' AND schemaname = 'temporal'
    ORDER BY indexname
""", conn)
print('Indexes on temporal.hyde_cells:')
print(idx_df.to_string(index=False) if len(idx_df) else '  NONE — build will be very slow!')

cols_df = pd.read_sql("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'public' AND table_name = 'basin08'
      AND column_name IN ('hybas_id', 'geom', 'sub_area')
    ORDER BY ordinal_position
""", conn)
print('\nbasin08 required columns:')
print(cols_df.to_string(index=False))

n_l08 = int(pd.read_sql("SELECT COUNT(*) AS n FROM public.basin08", conn).iloc[0].n)
n_l06 = int(pd.read_sql("SELECT COUNT(*) AS n FROM public.basin06", conn).iloc[0].n)
print(f'\nbasin06: {n_l06:,}   basin08: {n_l08:,}   ratio: {n_l08/n_l06:.1f}×')

l06_tables = pd.read_sql("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'temporal'
      AND table_name IN ('hyde_basin06_weights', 'hyde_basin06_steps', 'hyde_times')
    ORDER BY table_name
""", conn)
print('\nL06 reference tables:')
print(l06_tables.to_string(index=False))

n_steps = int(pd.read_sql("SELECT COUNT(*) AS n FROM temporal.hyde_times", conn).iloc[0].n)
print(f'\nhyde_times steps: {n_steps}')

# Check if L08 tables already exist
l08_check = pd.read_sql("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'temporal'
      AND table_name IN ('hyde_basin08_weights', 'hyde_basin08_steps')
    ORDER BY table_name
""", conn)
print('\nL08 tables already present:', l08_check['table_name'].tolist() if len(l08_check) else '(none — will build)')

Indexes on temporal.hyde_cells:
              indexname
        hyde_cells_pkey
idx_hyde_cells_centroid
    idx_hyde_cells_geom

basin08 required columns:
column_name        data_type
       geom     USER-DEFINED
   hybas_id double precision
   sub_area double precision

basin06: 16,397   basin08: 190,675   ratio: 11.6×

L06 reference tables:
          table_name
  hyde_basin06_steps
hyde_basin06_weights
          hyde_times

hyde_times steps: 128

L08 tables already present: (none — will build)


## 2 — Sample crosswalk timing

In [3]:
# Cell 3 — Time ST_Intersection on a sample of L08 basins
# L08 basins are small (avg ~19 HYDE cells vs L06's ~173), so per-basin cost is lower.
# Total crosswalk rows ~3.6M — only modestly larger than L06's 2.82M despite 11.6× basin count.

SAMPLE = 100
sql_sample = f"""
    WITH sample_basins AS (
        SELECT hybas_id, geom
        FROM public.basin08
        ORDER BY hybas_id
        LIMIT {SAMPLE}
    )
    SELECT
        b.hybas_id,
        h.cell_id,
        (ST_Area(ST_Intersection(h.geom, b.geom)) /
         NULLIF(ST_Area(h.geom), 0))::real AS overlap_frac
    FROM sample_basins b
    JOIN temporal.hyde_cells h ON ST_Intersects(h.geom, b.geom)
    WHERE ST_Area(ST_Intersection(h.geom, b.geom)) > 0
"""
t0 = time.time()
sample_xwalk = pd.read_sql(sql_sample, conn)
elapsed = time.time() - t0

t_per_basin  = elapsed / SAMPLE
est_rows     = len(sample_xwalk) / SAMPLE * n_l08
est_total_s  = t_per_basin * n_l08

print(f'{SAMPLE}-basin sample: {elapsed:.2f}s  →  {t_per_basin*1000:.1f} ms/basin')
print(f'Sample rows:           {len(sample_xwalk):,}  ({len(sample_xwalk)/SAMPLE:.1f} cells/basin avg)')
print(f'Estimated total rows:  {est_rows:,.0f}')
print(f'Estimated build time:  ~{est_total_s:.0f}s  ({est_total_s/60:.1f} min)')
print(f'\noverlap_frac: {sample_xwalk.overlap_frac.min():.4f} – {sample_xwalk.overlap_frac.max():.4f}')

100-basin sample: 0.15s  →  1.5 ms/basin
Sample rows:           1,910  (19.1 cells/basin avg)
Estimated total rows:  3,641,893
Estimated build time:  ~289s  (4.8 min)

overlap_frac: 0.0000 – 1.0000


## 3 — Build `temporal.hyde_basin08_weights`

One-time spatial materialization. Cell 3 gives the expected time. Skips if already built.

In [4]:
# Cell 4 — Build hyde_basin08_weights (one-time; skips if exists)
# Schema: (hybas_id bigint, cell_id integer, overlap_frac real)
# overlap_frac = planar ST_Area ratio — distortion cancels; denominator applied at query time.

exists = pd.read_sql("""
    SELECT EXISTS (
        SELECT 1 FROM information_schema.tables
        WHERE table_schema = 'temporal' AND table_name = 'hyde_basin08_weights'
    ) AS e
""", conn).iloc[0]['e']
print(f'temporal.hyde_basin08_weights exists: {exists}')

if not exists:
    print('Building crosswalk — see Cell 3 for estimated time…')
    t0 = time.time()
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE temporal.hyde_basin08_weights AS
            SELECT
                b.hybas_id,
                h.cell_id,
                (ST_Area(ST_Intersection(h.geom, b.geom)) /
                 NULLIF(ST_Area(h.geom), 0))::real AS overlap_frac
            FROM public.basin08 b
            JOIN temporal.hyde_cells h ON ST_Intersects(h.geom, b.geom)
            WHERE ST_Area(ST_Intersection(h.geom, b.geom)) > 0
        """)
    conn.commit()
    t_table = time.time()
    print(f'Table created: {t_table - t0:.0f}s  ({(t_table-t0)/60:.1f} min)')

    with conn.cursor() as cur:
        cur.execute("CREATE INDEX ON temporal.hyde_basin08_weights (hybas_id)")
        cur.execute("CREATE INDEX ON temporal.hyde_basin08_weights (cell_id)")
    conn.commit()
    t_done = time.time()
    print(f'Indexes built: {t_done - t_table:.0f}s')
    print(f'Total: {t_done - t0:.0f}s  ({(t_done - t0)/60:.1f} min)')
else:
    print('Already exists — skipping build.')

Indexes built: 3s
Total: 56s  (0.9 min)


## 4 — Verify crosswalk

In [5]:
# Cell 5 — Verify hyde_basin08_weights
stats = pd.read_sql("""
    SELECT
        COUNT(*)                  AS total_rows,
        COUNT(DISTINCT hybas_id)  AS n_basins,
        COUNT(DISTINCT cell_id)   AS n_cells,
        MIN(overlap_frac)         AS min_frac,
        MAX(overlap_frac)         AS max_frac
    FROM temporal.hyde_basin08_weights
""", conn).iloc[0]

n_covered  = int(stats.n_basins)
n_no_land  = n_l08 - n_covered

print(f'Total rows:          {int(stats.total_rows):>12,}')
print(f'Distinct basins:     {n_covered:>12,}   of {n_l08:,} total  ({n_no_land} no-land basins absent)')
print(f'Distinct HYDE cells: {int(stats.n_cells):>12,}   of 2,215,829 total')
print(f'overlap_frac:  min={stats.min_frac:.6f}   max={stats.max_frac:.6f}')
print(f'Mean cells/basin:    {int(stats.total_rows)/n_covered:.1f}   (L06 was 173.0)')

l06_rows = int(pd.read_sql("SELECT COUNT(*) AS n FROM temporal.hyde_basin06_weights", conn).iloc[0].n)
print(f'\nL06 crosswalk rows: {l06_rows:,}  (for comparison)')

sample = pd.read_sql(
    "SELECT hybas_id, cell_id, overlap_frac FROM temporal.hyde_basin08_weights LIMIT 8", conn
)
print('\nSample rows:')
print(sample.to_string(index=False))

Total rows:             4,306,122
Distinct basins:          189,850   of 190,675 total  (825 no-land basins absent)
Distinct HYDE cells:    2,215,285   of 2,215,829 total
overlap_frac:  min=0.000000   max=1.000000
Mean cells/basin:    22.7   (L06 was 173.0)

L06 crosswalk rows: 2,817,246  (for comparison)

Sample rows:
    hybas_id  cell_id  overlap_frac
4080179410.0  2176478      1.000000
4080053350.0  2176515      1.000000
3080725760.0  2236650      0.014883
3080725840.0  2236650      0.985117
4080054490.0  2236979      1.000000
4080220950.0  2267417      0.999880
4080236030.0  2267417      0.000120
4080234040.0  2267436      0.612782


## 5 — Build `temporal.hyde_basin08_steps`

Per-step loop over all 128 HYDE time steps. Each step inserts ~190k rows (one per covered
L08 basin). Progress is printed every 16 steps. Commits every 16 steps.

Expected time: scales with Cell 3's per-step output rows vs L06's 16k/step. If per-step time
is ~12× L06's ~4.6s → ~55s/step → ~120 min. On an M5 with fast local PostgreSQL, likely faster.

In [6]:
# Cell 6 — Build hyde_basin08_steps (one-time; skips if exists)
# Schema: (hybas_id bigint, step_idx smallint,
#           cropland_frac real, grazing_frac real, pasture_frac real, rangeland_frac real)
# Denominator: frac_full (÷ sub_area) — consistent with L06 (WO17/WO18 settled decision).

steps_exists = pd.read_sql("""
    SELECT EXISTS (
        SELECT 1 FROM information_schema.tables
        WHERE table_schema = 'temporal' AND table_name = 'hyde_basin08_steps'
    ) AS e
""", conn).iloc[0]['e']

if steps_exists:
    print('hyde_basin08_steps already exists — skipping build.')
else:
    ht = pd.read_sql(
        "SELECT step_idx, year_ce FROM temporal.hyde_times ORDER BY step_idx", conn
    )
    n_basins_in_xwalk = int(pd.read_sql(
        "SELECT COUNT(DISTINCT hybas_id) AS n FROM temporal.hyde_basin08_weights", conn
    ).iloc[0].n)
    print(f'Building over {len(ht)} steps × {n_basins_in_xwalk:,} basins…')
    print('Progress printed every 16 steps.\n')

    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE temporal.hyde_basin08_steps (
                hybas_id       bigint   NOT NULL,
                step_idx       smallint NOT NULL,
                cropland_frac  real,
                grazing_frac   real,
                pasture_frac   real,
                rangeland_frac real
            )
        """)
    conn.commit()

    t_total = time.time()
    BATCH = 16

    for i, row in ht.iterrows():
        step_idx = int(row.step_idx)
        pg_idx   = step_idx + 1
        year_ce  = int(row.year_ce)

        t0 = time.time()
        with conn.cursor() as cur:
            cur.execute(f"""
                INSERT INTO temporal.hyde_basin08_steps
                    (hybas_id, step_idx, cropland_frac, grazing_frac, pasture_frac, rangeland_frac)
                SELECT
                    w.hybas_id,
                    {step_idx},
                    SUM(h.cropland[{pg_idx}]  * w.overlap_frac) / NULLIF(MAX(b.sub_area), 0),
                    SUM(h.grazing[{pg_idx}]   * w.overlap_frac) / NULLIF(MAX(b.sub_area), 0),
                    SUM(h.pasture[{pg_idx}]   * w.overlap_frac) / NULLIF(MAX(b.sub_area), 0),
                    SUM(h.rangeland[{pg_idx}] * w.overlap_frac) / NULLIF(MAX(b.sub_area), 0)
                FROM temporal.hyde_basin08_weights w
                JOIN temporal.hyde_cells h  USING (cell_id)
                JOIN public.basin08        b USING (hybas_id)
                GROUP BY w.hybas_id
            """)
        if (step_idx + 1) % BATCH == 0 or step_idx == int(ht.step_idx.max()):
            conn.commit()

        elapsed = time.time() - t0
        if step_idx % 16 == 0 or step_idx < 3:
            total_so_far = time.time() - t_total
            pct = (i + 1) / len(ht) * 100
            est_remaining = (total_so_far / (i + 1)) * (len(ht) - i - 1)
            print(f'  step {step_idx:3d}  year {year_ce:+6d} CE  {elapsed:.1f}s  '
                  f'[{pct:.0f}%  {total_so_far:.0f}s elapsed  ~{est_remaining:.0f}s remaining]')

    t_insert = time.time() - t_total
    print(f'\nAll inserts done: {t_insert:.0f}s ({t_insert/60:.1f} min). Building indexes…')

    with conn.cursor() as cur:
        cur.execute("CREATE INDEX ON temporal.hyde_basin08_steps (step_idx)")
        cur.execute("CREATE INDEX ON temporal.hyde_basin08_steps (hybas_id, step_idx)")
    conn.commit()

    t_done = time.time() - t_total
    print(f'Indexes built. Total: {t_done:.0f}s ({t_done/60:.1f} min)')

Indexes built. Total: 1350s (22.5 min)


## 6 — Verify steps table

In [7]:
# Cell 7 — Verify hyde_basin08_steps
n_basins_xwalk = int(pd.read_sql(
    "SELECT COUNT(DISTINCT hybas_id) AS n FROM temporal.hyde_basin08_weights", conn
).iloc[0].n)
expected_rows = n_basins_xwalk * 128

stats = pd.read_sql("""
    SELECT
        COUNT(*)                  AS total_rows,
        COUNT(DISTINCT hybas_id)  AS n_basins,
        COUNT(DISTINCT step_idx)  AS n_steps
    FROM temporal.hyde_basin08_steps
""", conn).iloc[0]

print(f'Total rows:      {int(stats.total_rows):>12,}   (expected: {n_basins_xwalk:,} × 128 = {expected_rows:,})')
print(f'Distinct basins: {int(stats.n_basins):>12,}   (expected: {n_basins_xwalk:,})')
print(f'Distinct steps:  {int(stats.n_steps):>12,}   (expected: 128)')
print(f'Row count matches: {int(stats.total_rows) == expected_rows}')

sample = pd.read_sql(
    "SELECT * FROM temporal.hyde_basin08_steps WHERE step_idx = 20 LIMIT 8", conn
)
print('\nSample (step_idx=20, year 1000 CE):')
print(sample.to_string(index=False))

Total rows:        24,300,800   (expected: 189,850 × 128 = 24,300,800)
Distinct basins:      189,850   (expected: 189,850)
Distinct steps:           128   (expected: 128)
Row count matches: True

Sample (step_idx=20, year 1000 CE):
  hybas_id  step_idx  cropland_frac  grazing_frac  pasture_frac  rangeland_frac
2080063400        20            0.0           0.0           0.0             0.0
9080014410        20            0.0           0.0           0.0             0.0
3080035080        20            0.0           0.0           0.0             0.0
9080067740        20            0.0           0.0           0.0             0.0
9080024480        20            0.0           0.0           0.0             0.0
9080048080        20            0.0           0.0           0.0             0.0
9080041850        20            0.0           0.0           0.0             0.0
9080068920        20            0.0           0.0           0.0             0.0


## 7 — Per-request query timing — THE GO/NO-GO

Mirrors exactly what `/api/hyde/values?level=8` will execute after the route update.
Threshold: **<500ms** (L06 is 0.033s; L08 estimate ~0.38s).

In [8]:
# Cell 8 — Per-request timing
TARGET_YEAR = 1000
row = pd.read_sql(
    "SELECT step_idx FROM temporal.hyde_times WHERE year_ce <= %(y)s ORDER BY year_ce DESC LIMIT 1",
    conn, params={'y': TARGET_YEAR}
).iloc[0]
STEP = int(row.step_idx)
print(f'{TARGET_YEAR} CE → step_idx={STEP}')

# This is exactly what the route will run at level=8
sql_route = "SELECT hybas_id, cropland_frac FROM temporal.hyde_basin08_steps WHERE step_idx = %(s)s"

# Warm-up run
_ = pd.read_sql(sql_route, conn, params={'s': STEP})

times = []
for _ in range(3):
    t0 = time.time()
    result = pd.read_sql(sql_route, conn, params={'s': STEP})
    times.append(time.time() - t0)

mean_t = sum(times) / len(times)
print(f'Rows returned:        {len(result):,}')
print(f'Query times (3 runs): {[f"{t:.3f}s" for t in times]}')
print(f'Mean:                 {mean_t:.3f}s')
print(f'L06 baseline: 0.033s  |  Estimate: ~0.38s  |  Threshold: <0.500s')
print(f'\nGO / NO-GO: {"GO" if mean_t < 0.500 else "NO-GO — exceeds 500ms threshold"}')

Rows returned:        189,850
Query times (3 runs): ['0.403s', '0.393s', '0.410s']
Mean:                 0.402s
L06 baseline: 0.033s  |  Estimate: ~0.38s  |  Threshold: <0.500s

GO / NO-GO: GO


## 8 — Unit guard

L06 had one grazing/rangeland offender (hybas_id 5060271430, frac=1.0016). The route
clamps `min(v, 1.0)` before returning — same treatment will apply at L08.

In [ ]:
# Cell 9 — Unit guard: no fraction should exceed 1.0 by more than float32 noise
maxes = pd.read_sql("""
    SELECT
        MAX(cropland_frac)  AS max_cropland,
        MAX(grazing_frac)   AS max_grazing,
        MAX(pasture_frac)   AS max_pasture,
        MAX(rangeland_frac) AS max_rangeland
    FROM temporal.hyde_basin08_steps
    WHERE step_idx >= 10
""", conn).iloc[0]

print('Max fraction per variable (CE-era steps):')
over = []
for col, val in maxes.items():
    if val is None:
        print(f'  {col}: NULL')
        continue
    flag = ' *** OVER 1.0 ***' if val > 1.0 else ''
    print(f'  {col}: {val:.6f}{flag}')
    if val > 1.0:
        over.append(col)

if over:
    print(f'\nOffenders (up to 5 rows per variable):')
    for col in over:
        frac_col = col.replace('max_', '') + '_frac'
        df_off = pd.read_sql(
            f"SELECT hybas_id, {frac_col} FROM temporal.hyde_basin08_steps "
            f"WHERE {frac_col} > 1.0 ORDER BY {frac_col} DESC LIMIT 5",
            conn
        )
        print(f'  {frac_col}:')
        print(df_off.to_string(index=False))
    print('\nRoute will clamp min(v, 1.0) — same treatment as L06.')
else:
    print('\nUnit guard: PASSED — no fractions exceed 1.0')

## Summary

| Item | L06 actual | L08 actual |
|---|---|---|
| Crosswalk rows | 2,817,246 | 4,306,122 |
| Mean cells/basin | 173.0 | 22.7 |
| Basins covered | 16,281 / 16,397 | 189,850 / 190,675 |
| No-land basins | 116 | 825 |
| Steps rows | 2,083,968 | 24,300,800 |
| Crosswalk build time | 74s (1.2 min) | 56s (0.9 min) |
| Steps build time | 592s (9.9 min) | 1,350s (22.5 min) |
| Per-request time (DB query) | 0.033s | 0.402s |
| Per-request time (HTTP) | — | ~0.55s |
| Unit guard | grazing/rangeland: 1.0016 on one basin | (see Cell 9) |

**Stage 2b complete.** Route and UI wiring done:
- `/api/hyde/values` accepts `level=6|8`, dispatches to `temporal.hyde_basin0{level}_steps`.
- `sandbox_v3.html`: `#v3-level` and `#v3-polity-level` selects wired; all API calls thread level; \
  selective paint at L08 (scope members only); `loadBasinLayer()` level-aware; resets restore L06.